In [0]:
%sql
create Schema if not exists workspace.olist_gold

In [0]:
df_orders = spark.read.table("workspace.olist_silver.orders_customer")
df_payments = spark.read.table("workspace.olist_silver.orders_payment_totals")

display(df_orders.count(),'',df_payments.count())



In [0]:
paymante_validate_df = df_orders.join(df_payments, on = 'order_id', how = 'left')
display(paymante_validate_df.count())
#

In [0]:
print("Pedidos:", df_orders.count())
print("Pagamentos:", df_payments.count())
print("Após JOIN:", paymante_validate_df.count())

In [0]:
%sql
select * from workspace.olist_silver.orders_payment_totals
where order_id in ( "809a282bbd5dbcabb6f2f724fca862ec","
bfbd0f9bdef84302105ad712db648a6c")

In [0]:
%sql
select order_id, count(*) 
from workspace.olist_silver.orders_customer
group by order_id
having count(*) > 1 

In [0]:
paymante_validate_df.printSchema()

In [0]:
from pyspark.sql import functions as F

fact_order = paymante_validate_df.select(
    "order_id",
    "order_status",
    F.col("customer_unique_id").alias("customer_id"),
    F.col("order_purchase_timestamp").alias("order_date"),
    "total_payment",
    F.col("order_approved_at").alias("approved_at"),
    F.col("order_delivered_carrier_date").alias("shipped_at"),
    F.col("order_delivered_customer_date").alias("delivered_at"),
    F.to_date("order_estimated_delivery_date").alias("estimated_delivery_date"),
    F.upper(
        F.concat(
            F.substring("order_id", 1, 4),
            F.lit("-"),
            F.substring("order_id", -1, 1),
        )
    ).alias("order_id_shorter"),
)

counts = fact_order.agg(
    F.count("*").alias("rows"),
    F.countDistinct("order_id").alias("unique_orders"),
).first()

if counts["rows"] != counts["unique_orders"]:
    raise ValueError("Fact_order needs to have one row per order")

(
    fact_order.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.olist_gold.fact_order")
)

In [0]:
%sql
SELECT
    YEAR(order_date) AS order_year,
    COUNT(*) AS total_orders,
    COUNT_IF(order_status = 'delivered') AS delivered_orders,
    COUNT_IF(
        order_status = 'delivered' AND delivered_at IS NOT NULL
    ) AS orders_with_delivery_date,
    COUNT_IF(
        order_status = 'delivered'
        AND delivered_at IS NOT NULL
        AND estimated_delivery_date IS NOT NULL
    ) AS orders_eligible_for_sla,
    order_id_shorter
FROM workspace.olist_gold.fact_order
GROUP BY YEAR(order_date), order_id_shorter
ORDER BY order_year DESC;

In [0]:
%sql
select customer_id, count(*)
 from workspace.olist_gold.fact_order
 group by customer_id
 having count(*) > 1

In [0]:

from pyspark.sql import functions as F

average_order_value = (
    fact_order
    .groupBy("customer_id")
    .agg(
        F.sum("total_payment").alias("total_spend"),
        F.count("total_payment").alias("total_orders")
    )
    .withColumn("avg_order_value", 
                    F.col("total_spend") / F.col("total_orders")
                )
)
display(average_order_value)





In [0]:
average_order_value.count()

In [0]:
%sql
select count(distinct customer_id) from workspace.olist_gold.fact_order


In [0]:
average_order_value.write.mode("overwrite").saveAsTable("workspace.olist_gold.customer_order_metrics")